### PRE-ALGORITHM

#### 1 SECTION

In [ ]:
# Instalación de bibliotecas
!pip install scapy==2.5.0
!pip install joblib==1.4.2
!pip install scikit-learn==1.6.1
!pip install numpy==1.26.4
!pip install pandas==2.2.2
!pip install matplotlib==3.9.3
!pip install seaborn
!pip install deap
!pip install missingno
!pip install optuna
!pip install catboost==1.2.7
!pip install xgboost==2.1.3
!pip install lightgbm==4.5.0
!pip install tensorflow==2.18.0
!pip install shap
!pip install lime


In [ ]:
# =========================================
# Código Final ahora si para UNSW-15 en Google Colab
# =========================================

import os
import itertools
import time
import warnings
import math
import random
import datetime
import json

# Manipulación de datos y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno

# Machine learning y optimización
import optuna
import joblib
from joblib import dump, load

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    auc,
    roc_curve,
    mean_squared_error,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score
)

from sklearn.model_selection import (
    cross_val_score,
    cross_val_predict,
    train_test_split
)

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier

# Bibliotecas adicionales para validación y métricas
from sklearn import model_selection
from sklearn import metrics

# Boosting
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# Configuración de estilos de visualización // esto arroja error
# plt.style.use('seaborn')
# warnings.filterwarnings('ignore')

# Definir semillas para reproducibilidad
SEED_FIRST_TRAIN = 57


In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!ls /content/drive

In [ ]:
# Leer los archivos de entrenamiento y prueba
import os
data_dir = r"C:\Users\paulo\Downloads\TESIS\TESIS\Containers\Primero\IDS\Containers\Entrenamiento\Dataset\nb15"
data_dir = r'/content/drive/MyDrive/UNSW_NB15/'
# for dirname, _, filenames in os.walk('/content/drive/MyDrive/UNSW_NB15/'):
for dirname, _, filenames in os.walk(data_dir):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Ajustar las rutas según los archivos subidos
# train = pd.read_csv('/content/drive/MyDrive/UNSW_NB15/UNSW_NB15_training-set.csv')
# test = pd.read_csv('/content/drive/MyDrive/UNSW_NB15/UNSW_NB15_testing-set.csv')
train_path = os.path.join(data_dir, 'UNSW_NB15_training-set.csv')
test_path = os.path.join(data_dir, 'UNSW_NB15_testing-set.csv')
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [ ]:
# Mostrar las primeras filas del conjunto de prueba
test.head()

In [ ]:
# Concatenar los conjuntos de entrenamiento y prueba para preprocesarlos juntos
data = pd.concat([train, test]).reset_index(drop=True)
cols_cat = data.select_dtypes('object').columns
cols_numeric = data._get_numeric_data().columns

In [ ]:
# Verificar valores faltantes
print(data.isnull().sum())
missingno.matrix(data)

data['proto'].unique() # Esta es definitivamente una característica categórica.

data['service'].unique() # para tratar el tipo de servicio que es '-' y
# manejar valores faltantes o incorrectos
data['service'] = np.where(data['service'] == '-', 'None', data['service'])
print(data['service'].unique())

data['state'].unique() # se queda

# Automatizar el proceso de Manejar valores faltantes o incorrectos
def Remove_dump_values(data, cols):
    for col in cols:
        data[col] = np.where(data[col] == '-', 'None', data[col])
    return data

cols = data.columns
data_bin = Remove_dump_values(data, cols)

# Eliminar características innecesarias
data_bin = data.drop(['id'], axis=1)  # Eliminar columna 'id'
data_bin.drop(['attack_cat'], axis=1, inplace=True)
cols_cat = cols_cat.drop(['attack_cat'])

# Codificación one-hot para características categóricas
data_bin_hot = pd.get_dummies(data_bin, columns=cols_cat)

# Normalizar columnas numéricas
cols_numeric = list(cols_numeric)
cols_numeric.remove('label')
cols_numeric.remove('id')
data_bin_hot[cols_numeric] = data_bin_hot[cols_numeric].astype('float')
data_bin_hot[cols_numeric] = (data_bin_hot[cols_numeric] - np.min(data_bin_hot[cols_numeric])) / np.std(data_bin_hot[cols_numeric])

# Dividir en características (X) y etiqueta (Y)
X = data_bin_hot.drop('label', axis=1)
Y = data_bin_hot['label']

# Dividir en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=SEED_FIRST_TRAIN)

#### Guardar los modelos en la carpeta

In [ ]:

def guardar_modelo(modelo, nombre_modelo):
    """
    Función para guardar un modelo con el sufijo '_unsw' en la carpeta

    Args:
    modelo: El modelo entrenado que se quiere guardar.
    nombre_modelo: El nombre base del modelo (sin extensión ni sufijo).

    Returns:
    ruta_modelo: La ruta completa donde se ha guardado el modelo.
    """
    # Definir la ruta del directorio donde se guardarán los modelos
    modelo_dir = '/content/drive/MyDrive/Modelos'

    # Asegurarse de que el directorio existe, si no, crearlo
    if not os.path.exists(modelo_dir):
        os.makedirs(modelo_dir)

    # Agregar el sufijo '_unsw' al nombre del modelo
    modelo_nombre = f"{nombre_modelo}_unsw.pkl"

    # Generar la ruta completa
    modelo_ruta = os.path.join(modelo_dir, modelo_nombre)

    # Guardar el modelo utilizando joblib
    joblib.dump(modelo, modelo_ruta)

    print(f"Modelo guardado en: {modelo_ruta}")
    return modelo_ruta

# Función para cargar modelos
def cargar_modelo(nombre_modelo):
    modelo_dir = '/content/drive/MyDrive/Modelos'
    modelo_nombre = f'{nombre_modelo}_unsw.pkl'
    modelo_ruta = os.path.join(modelo_dir, modelo_nombre)
    modelo = joblib.load(modelo_ruta)
    print(f"Modelo {nombre_modelo} cargado correctamente desde {modelo_ruta}")
    return modelo


# def guardar_modelo(modelo, nombre_modelo):
#     """
#     Función para guardar un modelo con el sufijo '_unsw' en la carpeta.
#     Args:
#       modelo: El modelo entrenado que se quiere guardar.
#       nombre_modelo: El nombre base del modelo (sin extensión ni sufijo).

#     Returns:
#       ruta_modelo: La ruta completa donde se ha guardado el modelo.
#     """
#     # Definir la ruta del directorio donde se guardarán los modelos
#     modelo_dir = r"C:\Users\paulo\Downloads\TESIS"

#     # Asegurarse de que el directorio existe, si no, crearlo
#     if not os.path.exists(modelo_dir):
#         os.makedirs(modelo_dir)

#     # Agregar el sufijo '_unsw' al nombre del modelo
#     modelo_nombre = f"{nombre_modelo}_unsw.pkl"

#     # Generar la ruta completa
#     modelo_ruta = os.path.join(modelo_dir, modelo_nombre)

#     # Guardar el modelo utilizando joblib
#     joblib.dump(modelo, modelo_ruta)

#     print(f"Modelo guardado en: {modelo_ruta}")
#     return modelo_ruta


# def cargar_modelo(nombre_modelo):
#     """
#     Función para cargar un modelo (con sufijo '_unsw.pkl') desde
#     la carpeta C:\\Users\\paulo\\Downloads\\TESIS.

#     Args:
#       nombre_modelo: Nombre base del modelo (sin extensión ni sufijo).

#     Returns:
#       El modelo cargado en memoria.
#     """
#     modelo_dir = r"C:\Users\paulo\Downloads\TESIS"
#     modelo_nombre = f"{nombre_modelo}_unsw.pkl"
#     modelo_ruta = os.path.join(modelo_dir, modelo_nombre)

#     modelo = joblib.load(modelo_ruta)
#     print(f"Modelo {nombre_modelo} cargado correctamente desde {modelo_ruta}")
#     return modelo

# Conversión de las etiquetas a enteros
y_train = y_train.astype(int)
y_test = y_test.astype(int)


Notas <(ignorar)>:
*   marcos de aprendizaje automático automatizado (AutoML)
*   varios tipos de ataques, como preprocesamiento adversarial, envenenamiento y evasión
*   aprendizaje adversarial
*   teniendo en cuenta consideraciones de seguridad para el analisis ver los ataques de caja blanca siguiendo el principio de Kerckhoff y las mejores prácticas de seguridad el paper de doss.
*   Una evaluación de los aspectos adversariales no es un complemento, sino un componente obligatorio en la investigación de seguridad.

### ALGORITMOS

#### KNN

In [ ]:

# ------------------------------------------------------------------------------
# Rutas de archivos
# ------------------------------------------------------------------------------
# CHECKPOINT_PATH = '/content/drive/MyDrive/optuna_knn_study.pkl'
# TRAIN_TIME_PATH = '/content/drive/MyDrive/knn_train_time.txt'
# BEST_PARAMS_PATH = '/content/drive/MyDrive/best_hyperparameters_knn.json'
# MODEL_PATH = '/content/drive/MyDrive/knn_model_trained.pkl'


BASE_DIR = r"C:\Users\paulo\Downloads\TESIS\TESIS\Containers\Primero\IDS\Containers\Entrenamiento\Dataset\nb15\Checkpointws\UNSW15"

CHECKPOINT_PATH = os.path.join(BASE_DIR, "optuna_knn_study.pkl")
TRAIN_TIME_PATH = os.path.join(BASE_DIR, "knn_train_time.txt")
BEST_PARAMS_PATH = os.path.join(BASE_DIR, "best_hyperparameters_knn.json")
MODEL_PATH = os.path.join(BASE_DIR, "knn_model_trained.pkl")

# ------------------------------------------------------------------------------
# Función objetivo para Optuna
# ------------------------------------------------------------------------------
def objective_knn(trial):
    """Función objetivo que entrena y evalúa el modelo KNN
    con los hiperparámetros sugeridos"""
    n_neighbors = trial.suggest_int('n_neighbors', 2, 100)
    metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'chebyshev', 'minkowski'])
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])

    # Crear el clasificador con los parámetros sugeridos
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric, weights=weights)
    knn.fit(X_train, y_train)

    # Predecir y calcular la métrica en el conjunto de prueba
    y_pred = knn.predict(X_test)
    # En este caso, usamos F1-score promedio ponderado
    f1 = f1_score(y_test, y_pred, average='weighted')

    return f1  # Devolvemos el F1-score para que Optuna lo maximice

# ------------------------------------------------------------------------------
# Cargar (o crear) el estudio Optuna
# ------------------------------------------------------------------------------
try:
    # Intentar cargar un estudio previo desde el checkpoint
    study = load(CHECKPOINT_PATH)
    print(f"Checkpoint cargado. Trials completados: {len(study.trials)}")
except FileNotFoundError:
    # Si no existe el checkpoint, creamos un nuevo estudio
    study = optuna.create_study(direction='maximize')
    print("No se encontró un checkpoint. Creando un nuevo estudio.")

# ------------------------------------------------------------------------------
# Definir cuántos trials deseamos en total
# ------------------------------------------------------------------------------
desired_trials = 100
completed_trials = len(study.trials)
remaining_trials = max(0, desired_trials - completed_trials)

# ------------------------------------------------------------------------------
# Ejecutar los trials que faltan
# ------------------------------------------------------------------------------
if remaining_trials > 0:
    print(f"Ejecutando {remaining_trials} trials restantes...")
    study.optimize(
        objective_knn,
        n_trials=remaining_trials,
        # Con cada trial completado, se hace dump del estudio para no perder avances
        callbacks=[lambda study, trial: dump(study, CHECKPOINT_PATH)]
    )
else:
    print(f"Ya se completaron los {desired_trials} trials previstos.")

# ------------------------------------------------------------------------------
# Obtener los mejores hiperparámetros
# ------------------------------------------------------------------------------
best_params = study.best_trial.params
best_score = study.best_value  # Equivale a study.best_trial.value
print(f"Mejores hiperparámetros encontrados: {best_params}")
print(f"Mejor F1-score obtenido: {best_score}")

# Guardar los mejores hiperparámetros en un archivo JSON
with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=4)

print(f"Mejores hiperparámetros guardados en: {BEST_PARAMS_PATH}")

# ------------------------------------------------------------------------------
# Crear el modelo con los mejores hiperparámetros
# ------------------------------------------------------------------------------
best_knn = KNeighborsClassifier(**best_params)

# ------------------------------------------------------------------------------
# Entrenar el modelo final (si no está ya guardado)
# ------------------------------------------------------------------------------
try:
    # Si ya existe un modelo y un tiempo de entrenamiento guardados, cargarlos
    best_knn = load(MODEL_PATH)
    with open(TRAIN_TIME_PATH, 'r') as f:
        knn_train_time = float(f.read())
    print(f"Modelo cargado desde {MODEL_PATH}. Tiempo de entrenamiento previo: {knn_train_time} s")
except FileNotFoundError:
    # Si no existe el modelo entrenado, lo entrenamos, medimos el tiempo y lo guardamos
    print("No existe un modelo entrenado previo. Procediendo a entrenar...")
    start_time = time.time()
    best_knn.fit(X_train, y_train)
    end_time = time.time()
    knn_train_time = end_time - start_time
    print(f"Tiempo de entrenamiento: {knn_train_time} s")

    # Guardar el tiempo de entrenamiento
    with open(TRAIN_TIME_PATH, 'w') as f:
        f.write(str(knn_train_time))

# ------------------------------------------------------------------------------
# Medir el tiempo de predicción
# ------------------------------------------------------------------------------
# Comentado , Demora demasiado,: error:
# 592         def __exit__(self, type, value, traceback):
# 593         self.restore_original_limits()
#
# Código:
#
start_time = time.time()
y_pred = best_knn.predict(X_test)  # Aquí el modelo ya está entrenado
end_time = time.time()
knn_test_time = end_time - start_time
print(f"Tiempo de predicción: {knn_test_time} s")

# ------------------------------------------------------------------------------
# Cálculo de métricas finales
# ------------------------------------------------------------------------------
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
train_score = best_knn.score(X_train, y_train)
test_score = best_knn.score(X_test, y_test)

print(f"F1-Score en prueba: {f1*100:.2f}%")
print(f"Accuracy en prueba: {accuracy*100:.2f}%")
print(f"Puntuación en entrenamiento: {train_score}")
print(f"Puntuación en prueba: {test_score}")

# ------------------------------------------------------------------------------
# Mostrar algunas predicciones
# ------------------------------------------------------------------------------
y_test_pred_df = pd.DataFrame(y_pred, columns=['Prediction'])
print(y_test_pred_df.head())

print("\nProceso finalizado con éxito.")

# Guardar el modelo entrenado
guardar_modelo(best_knn, 'knn_model')

# Cargar el modelo y evaluarlo
loaded_knn_model = cargar_modelo('knn_model')
result = loaded_knn_model.score(X_test, y_test)
print(f"Precisión del modelo cargado: {result}")


Checkpoint cargado. Trials completados: 100
Ya se completaron los 100 trials previstos.
Mejores hiperparámetros encontrados: {'n_neighbors': 55, 'metric': 'manhattan', 'weights': 'distance'}
Mejor F1-score obtenido: 0.9350340224056298
Mejores hiperparámetros guardados en: C:\Users\paulo\Downloads\TESIS\TESIS\Containers\Primero\IDS\Containers\Entrenamiento\Dataset\nb15\Checkpointws\UNSW15\best_hyperparameters_knn.json
Modelo cargado desde C:\Users\paulo\Downloads\TESIS\TESIS\Containers\Primero\IDS\Containers\Entrenamiento\Dataset\nb15\Checkpointws\UNSW15\knn_model_trained.pkl. Tiempo de entrenamiento previo: 1.2482092380523682 s
Tiempo de predicción: 1760.9872877597809 s
F1-Score en prueba: 93.51%
Accuracy en prueba: 93.50%
Puntuación en entrenamiento: 0.9977435397042762
Puntuación en prueba: 0.9350340224056298
   Prediction
0           0
1           1
2           1
3           0
4           0

Proceso finalizado con éxito.
Modelo guardado en: C:\Users\paulo\Downloads\TESIS\knn_model_un

#### Logistic Regression

In [ ]:
def objective_logistic_regression(trial):
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear'])
    C = trial.suggest_float('C', 0.01, 100)
    model = LogisticRegression(solver=solver, C=C)
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    return accuracy

# Crear el estudio de Optuna para maximizar la precisión
study = optuna.create_study(direction='maximize')
study.optimize(objective_logistic_regression, n_trials=100)

# Obtener los mejores parámetros y la mejor puntuación de Optuna
best_params = study.best_trial.params
best_score = study.best_trial.value
print(f"Mejores Hiperparámetros: {best_params}")
print(f"Mejor Puntuación: {best_score}")

# Entrenar el modelo final con los mejores hiperparámetros encontrados por Optuna
lg_model = LogisticRegression(solver=best_params['solver'], C=best_params['C'])

# Medir el tiempo de entrenamiento
start_time = time.time()
lg_model.fit(X_train, y_train)
end_time = time.time()
training_time = end_time - start_time
print(f"Tiempo de entrenamiento: {training_time} segundos")

# Medir el tiempo de predicción
start_time = time.time()
y_test_pred = lg_model.predict(X_test)
end_time = time.time()
prediction_time = end_time - start_time
print(f"Tiempo de predicción: {prediction_time} segundos")

# Evaluar el modelo en los conjuntos de entrenamiento y prueba
train_score = lg_model.score(X_train, y_train)
test_score = lg_model.score(X_test, y_test)
print(f"Precisión en el conjunto de entrenamiento: {train_score}")
print(f"Precisión en el conjunto de prueba: {test_score}")

# Guardar las predicciones como DataFrame
y_test_pred_df = pd.DataFrame(y_test_pred, columns=['Prediction'])
print("Primeras predicciones del conjunto de prueba:")
print(y_test_pred_df.head())

# Guardar el modelo de regresión logística entrenado en la ruta especificada con el sufijo '_unsw'
modelo_ruta = guardar_modelo(lg_model, 'logistic_regression_model')

print(f"Modelo guardado en: {modelo_ruta}")

# Cargar el modelo guardado para futuras pruebas
loaded_model = joblib.load(modelo_ruta)
result = loaded_model.score(X_test, y_test)
print(f"Precisión del modelo cargado: {result}")


[I 2025-02-19 08:16:25,450] A new study created in memory with name: no-name-e42c5089-6fa7-4405-bcb7-6d797180c1c4
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[I 2025-02-19 08:16:31,321] Trial 0 finished with value: 0.9026027787120644 and parameters: {'solver': 'lbfgs', 'C': 42.034309579271266}. Best is trial 0 with value: 0.9026027787120644.
[I 2025-02-19 08:17:40,915] Trial 1 finished with value: 0.9057851025846679 and parameters: {'solver': 'liblinear', 'C': 12.846854252314051}. Best is trial 1 with value: 0.9057851025846679.

Mejores Hiperparámetros: {'solver': 'liblinear', 'C': 90.8277299142791}
Mejor Puntuación: 0.9067553232775348
Tiempo de entrenamiento: 105.3629367351532 segundos
Tiempo de predicción: 0.06789779663085938 segundos
Precisión en el conjunto de entrenamiento: 0.9059715807973565
Precisión en el conjunto de prueba: 0.9067553232775348
Primeras predicciones del conjunto de prueba:
   Prediction
0           0
1           1
2           1
3           0
4           0
Modelo guardado en: /content/drive/MyDrive/Modelos/logistic_regression_model_unsw.pkl
Modelo guardado en: /content/drive/MyDrive/Modelos/logistic_regression_model_unsw.pkl
Precisión del modelo cargado: 0.9067553232775348


#### Decision Tree

In [ ]:
# Definir el objetivo para Optuna
def objective_decision_tree(trial):
    # Sugerir valores para los hiperparámetros
    max_depth = trial.suggest_int('max_depth', 3, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    max_features = trial.suggest_int('max_features', 1, X_train.shape[1])  # Máximo de features basado en el dataset

    # Configurar el modelo con los hiperparámetros sugeridos
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Retornar la puntuación en el conjunto de prueba
    return model.score(X_test, y_test)

# Crear el estudio para maximizar la precisión
study = optuna.create_study(direction='maximize')
study.optimize(objective_decision_tree, n_trials=100)

# Obtener los mejores hiperparámetros y la mejor puntuación
best_params = study.best_trial.params
best_score = study.best_trial.value
print(f"Mejores Hiperparámetros: {best_params}")
print(f"Mejor Puntuación: {best_score}")

# Entrenar el modelo final con los mejores hiperparámetros
dt = DecisionTreeClassifier(
    max_features=best_params['max_features'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    random_state=42
)

# Medir el tiempo de entrenamiento
start_time = time.time()
dt.fit(X_train, y_train)
end_time = time.time()
training_time = end_time - start_time
print(f"Tiempo de entrenamiento: {training_time} segundos")

# Medir el tiempo de predicción
start_time = time.time()
y_pred2 = dt.predict(X_test)
end_time = time.time()
prediction_time = end_time - start_time
print(f"Tiempo de predicción: {prediction_time} segundos")

# Evaluar y mostrar los resultados
train_score = dt.score(X_train, y_train)
test_score = dt.score(X_test, y_test)
print(f"Precisión en el conjunto de entrenamiento: {train_score}")
print(f"Precisión en el conjunto de prueba: {test_score}")

# Guardar el modelo entrenado
guardar_modelo(dt, 'decision_tree_model')

# Cargar el modelo guardado y evaluarlo
loaded_dt_model = cargar_modelo('decision_tree_model')
result = loaded_dt_model.score(X_test, y_test)
print(f"Precisión del modelo cargado: {result}")


[I 2025-02-19 08:09:38,583] A new study created in memory with name: no-name-d4faccf0-9108-4bb6-9bb1-3a86e88a527c
[I 2025-02-19 08:09:41,212] Trial 0 finished with value: 0.9372590618612714 and parameters: {'max_depth': 27, 'min_samples_split': 7, 'max_features': 113}. Best is trial 0 with value: 0.9372590618612714.
[I 2025-02-19 08:09:43,180] Trial 1 finished with value: 0.9195622364233784 and parameters: {'max_depth': 6, 'min_samples_split': 4, 'max_features': 153}. Best is trial 0 with value: 0.9372590618612714.
[I 2025-02-19 08:09:44,581] Trial 2 finished with value: 0.9396910817313912 and parameters: {'max_depth': 16, 'min_samples_split': 6, 'max_features': 69}. Best is trial 2 with value: 0.9396910817313912.
[I 2025-02-19 08:09:45,760] Trial 3 finished with value: 0.9360559882021163 and parameters: {'max_depth': 28, 'min_samples_split': 8, 'max_features': 51}. Best is trial 2 with value: 0.9396910817313912.
[I 2025-02-19 08:09:48,671] Trial 4 finished with value: 0.94013091511215

Mejores Hiperparámetros: {'max_depth': 18, 'min_samples_split': 3, 'max_features': 169}
Mejor Puntuación: 0.9417738221520788
Tiempo de entrenamiento: 4.036796569824219 segundos
Tiempo de predicción: 0.027887344360351562 segundos
Precisión en el conjunto de entrenamiento: 0.9650054609665634
Precisión en el conjunto de prueba: 0.9417738221520788
Modelo guardado en: /content/drive/MyDrive/Modelos/decision_tree_model_unsw.pkl
Modelo decision_tree_model cargado correctamente desde /content/drive/MyDrive/Modelos/decision_tree_model_unsw.pkl
Precisión del modelo cargado: 0.9417738221520788


In [ ]:
###
## Ejemplo realizando el modelo con todos los hiperparametros del modelo.
## NO SE CONSIDERO , Código no usado
####

import optuna
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Definir el objetivo para Optuna
def objective_decision_tree(trial):
    # Sugerir valores para los hiperparámetros
    max_depth = trial.suggest_int('max_depth', 3, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_int('max_features', 1, X_train.shape[1])
    min_impurity_decrease = trial.suggest_float('min_impurity_decrease', 0.0, 0.1)
    ccp_alpha = trial.suggest_float('ccp_alpha', 0.0, 0.1)
    max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 10, 1000)
    min_weight_fraction_leaf = trial.suggest_float('min_weight_fraction_leaf', 0.0, 0.5)
    splitter = trial.suggest_categorical('splitter', ['best', 'random'])
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])

    # Configurar el modelo con los hiperparámetros sugeridos
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        min_impurity_decrease=min_impurity_decrease,
        ccp_alpha=ccp_alpha,
        max_leaf_nodes=max_leaf_nodes,
        min_weight_fraction_leaf=min_weight_fraction_leaf,
        splitter=splitter,
        criterion=criterion,
        class_weight=class_weight,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Evaluar el modelo
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy

# Crear el estudio para maximizar la precisión
study = optuna.create_study(direction='maximize')
study.optimize(objective_decision_tree, n_trials=100)

# Obtener los mejores hiperparámetros
best_params = study.best_trial.params
print(f"Mejores Hiperparámetros: {best_params}")

# Entrenar el modelo final con los mejores hiperparámetros
dt = DecisionTreeClassifier(**best_params)

start_time = time.time()
dt.fit(X_train, y_train)
training_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {training_time} segundos")

y_pred = dt.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión en el conjunto de prueba: {accuracy}")


[I 2025-02-19 08:14:53,548] A new study created in memory with name: no-name-394176a0-11b1-47dd-86a4-3d10c86d578a
[I 2025-02-19 08:14:53,743] Trial 0 finished with value: 0.7343147654653178 and parameters: {'max_depth': 18, 'min_samples_split': 16, 'min_samples_leaf': 1, 'max_features': 177, 'min_impurity_decrease': 0.09108620899369278, 'ccp_alpha': 0.058227628295154925, 'max_leaf_nodes': 467, 'min_weight_fraction_leaf': 0.3582745542033611, 'splitter': 'random', 'criterion': 'entropy', 'class_weight': None}. Best is trial 0 with value: 0.7343147654653178.
[I 2025-02-19 08:14:54,460] Trial 1 finished with value: 0.8134071563478306 and parameters: {'max_depth': 17, 'min_samples_split': 17, 'min_samples_leaf': 1, 'max_features': 185, 'min_impurity_decrease': 0.012537937293385272, 'ccp_alpha': 0.055248374270511325, 'max_leaf_nodes': 120, 'min_weight_fraction_leaf': 0.2521895402308508, 'splitter': 'best', 'criterion': 'entropy', 'class_weight': None}. Best is trial 1 with value: 0.813407156

Mejores Hiperparámetros: {'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 10, 'max_features': 52, 'min_impurity_decrease': 0.021541324036172982, 'ccp_alpha': 0.010679809297959355, 'max_leaf_nodes': 696, 'min_weight_fraction_leaf': 0.10863345238776467, 'splitter': 'best', 'criterion': 'gini', 'class_weight': 'balanced'}
Tiempo de entrenamiento: 0.3965778350830078 segundos
Precisión en el conjunto de prueba: 0.8715686528162273


#### XGB

In [ ]:

# Conversión de las etiquetas a enteros
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Usar Optuna para optimizar el modelo XGBoost
def objective(trial):
    # Definir los parámetros a optimizar
    n_estimators = trial.suggest_int('n_estimators', 50, 128)
    max_depth = trial.suggest_int('max_depth', 2, 6)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1, log=True)
    subsample = trial.suggest_categorical('subsample', [0.5, 0.8])
    colsample_bytree = trial.suggest_categorical('colsample_bytree', [0.5, 0.8])

    # Inicializar el modelo
    XGB = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=42
    )

    # Entrenar el modelo
    XGB.fit(X_train, y_train)

    # Evaluar el modelo
    y_pred = XGB.predict(X_test)
    f1 = f1_score(y_test, y_pred)

    return f1

# Inicializar y optimizar el estudio de Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Obtener los mejores hiperparámetros y la mejor puntuación
best_score = study.best_value
best_params = study.best_trial.params

# Entrenar el modelo final con los mejores hiperparámetros
XGB_model = XGBClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    random_state=42
)

# Medir el tiempo de entrenamiento
start_time = time.time()
XGB_model.fit(X_train, y_train)
end_time = time.time()
cld_train = end_time - start_time
print("Training time:", cld_train)

# Medir el tiempo de predicción
start_time = time.time()
y_pred = XGB_model.predict(X_test)
end_time = time.time()
cld_test = end_time - start_time
print("Testing time:", cld_test)

# Evaluar y mostrar los resultados
print(f"F1 Score: {f1_score(y_test, y_pred) * 100:.2f}%")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")

CLF_train, CLF_test = XGB_model.score(X_train, y_train), XGB_model.score(X_test, y_test)
print(f"Train Score: {CLF_train}")
print(f"Test Score: {CLF_test}")

# Imprimir los mejores hiperparámetros
print("Best hyperparameters:", best_params)
print("Best score:", best_score)

# Guardar el modelo entrenado
modelo_ruta = guardar_modelo(XGB_model, 'XGB_model')

# Cargar el modelo guardado para futuras pruebas
loaded_XGB = joblib.load(modelo_ruta)
print(f"Precisión del modelo cargado: {loaded_XGB.score(X_test, y_test)}")


[I 2025-02-19 10:14:21,135] A new study created in memory with name: no-name-0183f643-1207-4867-8242-c1cadecc25d6
[I 2025-02-19 10:14:25,281] Trial 0 finished with value: 0.9281188551452917 and parameters: {'n_estimators': 72, 'max_depth': 2, 'learning_rate': 0.05354168292577245, 'subsample': 0.5, 'colsample_bytree': 0.5}. Best is trial 0 with value: 0.9281188551452917.
[I 2025-02-19 10:14:29,057] Trial 1 finished with value: 0.9280549666868565 and parameters: {'n_estimators': 68, 'max_depth': 2, 'learning_rate': 0.056100704957702266, 'subsample': 0.8, 'colsample_bytree': 0.8}. Best is trial 0 with value: 0.9281188551452917.
[I 2025-02-19 10:14:36,202] Trial 2 finished with value: 0.9481004481408212 and parameters: {'n_estimators': 76, 'max_depth': 6, 'learning_rate': 0.030125674530688887, 'subsample': 0.5, 'colsample_bytree': 0.5}. Best is trial 2 with value: 0.9481004481408212.
[I 2025-02-19 10:14:39,814] Trial 3 finished with value: 0.9096128367437067 and parameters: {'n_estimators'

Training time: 6.989834308624268
Testing time: 0.17878031730651855
F1 Score: 94.81%
Accuracy Score: 93.35%
Train Score: 0.9329659424186815
Test Score: 0.9334816692970428
Best hyperparameters: {'n_estimators': 76, 'max_depth': 6, 'learning_rate': 0.030125674530688887, 'subsample': 0.5, 'colsample_bytree': 0.5}
Best score: 0.9481004481408212
Modelo guardado en: /content/drive/MyDrive/Modelos/XGB_model_unsw.pkl
Precisión del modelo cargado: 0.9334816692970428


#### GNB

In [ ]:

# DEAP
from deap import base, creator, tools, algorithms
# prototipo de funcion cargar y guardar, version 0
def guardar_modelo(modelo, nombre_archivo):
    joblib.dump(modelo, f"{nombre_archivo}.pkl")
    print(f"[Info] Modelo guardado en {nombre_archivo}.pkl")

def cargar_modelo(nombre_archivo):
    mod = joblib.load(f"{nombre_archivo}.pkl")
    print(f"[Info] Modelo cargado desde {nombre_archivo}.pkl")
    return mod

# Preparación DEAP (Algoritmo genético)
# Crea las clases si no existen
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Definir el rango de los parámetros (GA)
toolbox.register("var_smoothing", random.uniform, 1e-10, 1e-1)

# Crear individuo y población
toolbox.register("individual", tools.initCycle, creator.Individual, (toolbox.var_smoothing,), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

#===
# Función de evaluación del individuo (GA)
#===
def evaluate(individual):
    """
    Recibe 'individual' con [var_smoothing].
    Entrena un GaussianNB y retorna la precisión en X_test, y_test (UNSW).
    """
    var_smoothing = max(1e-10, individual[0])  # se asegura que sea > 0
    gnb = GaussianNB(var_smoothing=var_smoothing)
    gnb.fit(X_train, y_train)
    y_pred = gnb.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return (accuracy,)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)

#===
# Mutación para mantener valores positivos
#===
def mut_positive(individual, mu, sigma, indpb):
    """
    Añade una perturbación gaussiana y asegura var_smoothing >= 1e-10.
    """
    for i in range(len(individual)):
        if random.random() < indpb:
            individual[i] += random.gauss(mu, sigma)
            individual[i] = max(1e-10, individual[i])
    return (individual,)

toolbox.register("mutate", mut_positive, mu=0, sigma=1e-2, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

#===
# Algoritmo Clonal con Optuna
#===
def objective(trial):
    """
    Función objetivo para Optuna.
    Se optimiza 'var_smoothing' en [1e-10, 1e-1].
    """
    var_smoothing = trial.suggest_float('var_smoothing', 1e-10, 1e-1, log=True)
    gnb = GaussianNB(var_smoothing=var_smoothing)
    gnb.fit(X_train, y_train)
    y_pred = gnb.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

#===
# Parámetros del Algoritmo Genético
#===
N_POP = 10
N_GEN = 10
CXPB = 0.5
MUTPB = 0.2

#===
# Inicialización de la Población
#===
population = toolbox.population(n=N_POP)

#===
# Bucle de Evolución (GA) + Optuna en cada generación
#===
for gen in range(N_GEN):
    # Variar y Evaluar
    offspring = algorithms.varAnd(population, toolbox, CXPB, MUTPB)
    for ind in offspring:
        ind[0] = max(1e-10, ind[0])  # Clampeo
    fits = [toolbox.evaluate(ind) for ind in offspring]
    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit

    # Seleccionar siguiente gen
    population = toolbox.select(offspring, k=len(population))

    # Corre Optuna en cada generación
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=5)

    # Tomar el mejor de la población (GA)
    best_individual = tools.selBest(population, k=1)[0]
    # Tomar el mejor de Optuna
    best_params = study.best_trial.params
    # Actualizar individuo con el var_smoothing de Optuna
    best_individual[0] = best_params['var_smoothing']

#===
# Entrenamiento final con el mejor modelo
#===
best_var_smoothing = population[0][0]  # O selBest population
gnb = GaussianNB(var_smoothing=best_var_smoothing)

# Tiempo de entrenamiento
start_time = time.time()
gnb.fit(X_train, y_train)
end_time = time.time()
gnb_train_time = end_time - start_time
print("Training time:", gnb_train_time)

# Tiempo de predicción
start_time = time.time()
y_pred = gnb.predict(X_test)
end_time = time.time()
gnb_test_time = end_time - start_time
print("Testing time:", gnb_test_time)

# Evaluación final
acc_train = gnb.score(X_train, y_train)
acc_test  = gnb.score(X_test, y_test)
print(f"Train Score: {acc_train}")
print(f"Test Score:  {acc_test}")

print("[Info] Mejor var_smoothing:", best_var_smoothing)

# Guardar y cargar el modelo
guardar_modelo(gnb, "gnb_model_unsw")
loaded_gnb = cargar_modelo("gnb_model_unsw")
print(f"Precisión del modelo cargado: {loaded_gnb.score(X_test, y_test)}")


[I 2025-02-19 10:15:30,528] A new study created in memory with name: no-name-78e2c4b9-6f27-4de5-ab06-19e4c7e0d4d5
[I 2025-02-19 10:15:31,344] Trial 0 finished with value: 0.5040749269100412 and parameters: {'var_smoothing': 1.2706856570097077e-09}. Best is trial 0 with value: 0.5040749269100412.
[I 2025-02-19 10:15:32,137] Trial 1 finished with value: 0.7541202038757082 and parameters: {'var_smoothing': 0.021959104677990164}. Best is trial 1 with value: 0.7541202038757082.
[I 2025-02-19 10:15:32,795] Trial 2 finished with value: 0.5040231818064216 and parameters: {'var_smoothing': 3.6959128148070115e-10}. Best is trial 1 with value: 0.7541202038757082.
[I 2025-02-19 10:15:33,422] Trial 3 finished with value: 0.5041784171172803 and parameters: {'var_smoothing': 2.8431328025593195e-09}. Best is trial 1 with value: 0.7541202038757082.
[I 2025-02-19 10:15:34,042] Trial 4 finished with value: 0.50420428966909 and parameters: {'var_smoothing': 3.2829122274935286e-09}. Best is trial 1 with va

Training time: 0.4705777168273926
Testing time: 0.15050983428955078
Train Score: 0.7553764186038776
Test Score:  0.7569403120229748
[Info] Mejor var_smoothing: 0.012659964795831717
[Info] Modelo guardado en gnb_model_unsw.pkl
[Info] Modelo cargado desde gnb_model_unsw.pkl
Precisión del modelo cargado: 0.7569403120229748


#### Gradient Boosting Machines (LightGBM)



In [ ]:

# Definir la función de optimización con Optuna
def objective_lightgbm(trial):
    param = {
        'objective': 'binary',            # Tipo de problema: clasificación binaria
        'metric': 'binary_logloss',       # Métrica a optimizar
        'boosting_type': 'gbdt',          # Tipo de boosting (Gradient Boosting Decision Trees)
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),  # Tasa de aprendizaje
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),                     # Número máximo de hojas por árbol
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),      # Fracción de características
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),      # Fracción de datos para bagging
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),                    # Frecuencia de bagging
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100)         # Muestras mínimas en nodo hoja
    }

    # Crear modelo LightGBM Classifier
    model = LGBMClassifier(**param, random_state=42)

    # Medir tiempo de entrenamiento
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    # Medir tiempo de predicción
    start_time = time.time()
    preds = model.predict(X_test)  # Devuelve clases (0 o 1)
    pred_time = time.time() - start_time

    accuracy = accuracy_score(y_test, preds)

    print(f"Training time: {train_time:.4f} seconds")
    print(f"Prediction time: {pred_time:.4f} seconds")
    return accuracy

# Crear y optimizar el modelo con Optuna
study_lightgbm = optuna.create_study(direction='maximize')
study_lightgbm.optimize(objective_lightgbm, n_trials=10)

# Obtener los mejores hiperparámetros
best_params_lightgbm = study_lightgbm.best_trial.params
print(f"Best Hyperparameters for LightGBM: {best_params_lightgbm}")

# Entrenar modelo final con los mejores hiperparámetros
final_model = LGBMClassifier(**best_params_lightgbm, random_state=42)

start_time = time.time()
final_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Final Training time: {train_time:.4f} seconds")

# Evaluar el modelo final en el conjunto de prueba
start_time = time.time()
final_preds = final_model.predict(X_test)
pred_time = time.time() - start_time

# Calcular métricas
final_accuracy = accuracy_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds)
final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
final_roc_auc = roc_auc_score(y_test, final_preds)

print(f"Final Test Accuracy for LightGBM: {final_accuracy:.4f}")
print(f"Final F1 Score: {final_f1:.4f}")
print(f"Final Precision: {final_precision:.4f}")
print(f"Final Recall: {final_recall:.4f}")
print(f"Final ROC-AUC Score: {final_roc_auc:.4f}")
print(f"Final Prediction time: {pred_time:.4f} seconds")

# Guardar el modelo entrenado
guardar_modelo(final_model, 'lightgbm_model')

# Cargar el modelo guardado
loaded_gbm_model = cargar_modelo('lightgbm_model')

# Cross-Validation (Validación Cruzada)
from sklearn.model_selection import cross_val_score
scores = cross_val_score(final_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Cross-Validation Accuracy: {scores.mean():.4f}")


[I 2025-03-05 16:03:38,898] A new study created in memory with name: no-name-a71fcc93-f045-4961-a5ef-42bce3602ba7


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.531443801471317, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.531443801471317
[LightGBM] [Warning] bagging_fraction is set=0.8886355751428487, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8886355751428487
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.531443801471317, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.531443801471317
[LightGBM] [Warning] bagging_fraction is set=0.8886355751428487, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8886355751428487
[LightGBM] [Info] Number of positive: 115241, number of negative: 65130
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.210612 seconds.
You can se

[I 2025-03-05 16:03:49,086] Trial 0 finished with value: 0.9497296318335877 and parameters: {'learning_rate': 0.05898345914025315, 'num_leaves': 133, 'feature_fraction': 0.531443801471317, 'bagging_fraction': 0.8886355751428487, 'bagging_freq': 5, 'min_child_samples': 33}. Best is trial 0 with value: 0.9497296318335877.


Training time: 9.0800 seconds
Prediction time: 1.0934 seconds
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.9546875909142832, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9546875909142832
[LightGBM] [Warning] bagging_fraction is set=0.7608831487621426, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7608831487621426
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.9546875909142832, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9546875909142832
[LightGBM] [Warning] bagging_fraction is set=0.7608831487621426, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7608831487621426
[LightGBM] [Info] Number of positive: 115241, number of negative: 65130
[LightGBM] [Info] Auto-choosing row-wise multi-t

[I 2025-03-05 16:03:59,260] Trial 1 finished with value: 0.9500271661794003 and parameters: {'learning_rate': 0.18773122777637588, 'num_leaves': 197, 'feature_fraction': 0.9546875909142832, 'bagging_fraction': 0.7608831487621426, 'bagging_freq': 4, 'min_child_samples': 41}. Best is trial 1 with value: 0.9500271661794003.


...

Training time: 7.2597 seconds
Prediction time: 0.6616 seconds
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.5732597012849953, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5732597012849953
[LightGBM] [Warning] bagging_fraction is set=0.5535366145554461, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5535366145554461
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.5732597012849953, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5732597012849953
[LightGBM] [Warning] bagging_fraction is set=0.5535366145554461, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5535366145554461
[LightGBM] [Info] Number of positive: 115241, number of negative: 65130
[LightGBM] [Info] Auto-choosing row-wise multi-t

[I 2025-03-05 16:04:58,212] Trial 8 finished with value: 0.9480996610695713 and parameters: {'learning_rate': 0.15114080533960628, 'num_leaves': 94, 'feature_fraction': 0.5732597012849953, 'bagging_fraction': 0.5535366145554461, 'bagging_freq': 4, 'min_child_samples': 86}. Best is trial 1 with value: 0.9500271661794003.


Training time: 7.2515 seconds
Prediction time: 1.0813 seconds
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.7125993427631603, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7125993427631603
[LightGBM] [Warning] bagging_fraction is set=0.6068624388661048, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6068624388661048
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.7125993427631603, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7125993427631603
[LightGBM] [Warning] bagging_fraction is set=0.6068624388661048, subsample=1.0 will be ignored. Current value: bagging_fraction=0.6068624388661048
[LightGBM] [Info] Number of positive: 115241, number of negative: 65130
[LightGBM] [Info] Auto-choosing row-wise multi-t

[I 2025-03-05 16:05:05,225] Trial 9 finished with value: 0.9483713228635741 and parameters: {'learning_rate': 0.0782614902855027, 'num_leaves': 112, 'feature_fraction': 0.7125993427631603, 'bagging_fraction': 0.6068624388661048, 'bagging_freq': 1, 'min_child_samples': 78}. Best is trial 1 with value: 0.9500271661794003.


Training time: 6.2010 seconds
Prediction time: 0.8010 seconds
Best Hyperparameters for LightGBM: {'learning_rate': 0.18773122777637588, 'num_leaves': 197, 'feature_fraction': 0.9546875909142832, 'bagging_fraction': 0.7608831487621426, 'bagging_freq': 4, 'min_child_samples': 41}
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.9546875909142832, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9546875909142832
[LightGBM] [Warning] bagging_fraction is set=0.7608831487621426, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7608831487621426
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.9546875909142832, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9546875909142832
[LightGBM] [Warning] bagging_fraction is set=0.760

#### Gradient Boosting Machines (CatBoost)

In [ ]:
def objective_catboost(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 100, 1000),  # Número de iteraciones
        'depth': trial.suggest_int('depth', 4, 10),  # Profundidad de los árboles
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),  # Tasa de aprendizaje
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),  # Regularización L2
        'border_count': trial.suggest_int('border_count', 32, 255),  # Número de umbrales para cuantización
    }

    model = CatBoostClassifier(**param, verbose=0)  # Instanciación del modelo con los parámetros sugeridos

    # Medir el tiempo de entrenamiento
    start_time = time.time()
    model.fit(X_train, y_train)  # Entrenamiento del modelo
    train_time = time.time() - start_time

    # Medir el tiempo de predicción
    start_time = time.time()
    preds = model.predict(X_test)  # Predicciones en el conjunto de test
    pred_time = time.time() - start_time

    accuracy = accuracy_score(y_test, preds)  # Cálculo de la precisión
    print(f"Training time: {train_time} seconds")
    print(f"Prediction time: {pred_time} seconds")
    return accuracy  # Retorno de la métrica a optimizar

# Crear el estudio para CatBoost
study_catboost = optuna.create_study(direction='maximize')  # Crear un estudio para maximizar la precisión
study_catboost.optimize(objective_catboost, n_trials=100)  # Ejecutar la optimización con 100 pruebas

# Obtener los mejores hiperparámetros
best_params_catboost = study_catboost.best_trial.params
print(f"Best Hyperparameters for CatBoost: {best_params_catboost}")

# Entrenar el modelo final con los mejores hiperparámetros
final_catboost = CatBoostClassifier(**best_params_catboost, verbose=0)

start_time = time.time()
final_catboost.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Final Training time: {train_time} seconds")

# Evaluar el modelo final en el conjunto de prueba
start_time = time.time()
final_preds = final_catboost.predict(X_test)
pred_time = time.time() - start_time
final_accuracy = accuracy_score(y_test, final_preds)
print(f"Final Test Accuracy for CatBoost: {final_accuracy}")
print(f"Final Prediction time: {pred_time} seconds")

# Guardar el modelo entrenado
guardar_modelo(final_catboost, 'catboost_model')

# Cargar el modelo guardado
loaded_gcm_model = cargar_modelo('catboost_model')


[I 2025-03-05 11:10:21,098] A new study created in memory with name: no-name-219a6e4e-7072-4a44-b31c-cefa1b5f1326
[I 2025-03-05 11:11:00,959] Trial 0 finished with value: 0.9479185532069028 and parameters: {'iterations': 561, 'depth': 6, 'learning_rate': 0.1347824449142597, 'l2_leaf_reg': 0.0010430994757921002, 'border_count': 88}. Best is trial 0 with value: 0.9479185532069028.


Training time: 39.74458718299866 seconds
Prediction time: 0.10459399223327637 seconds


[I 2025-03-05 11:11:55,243] Trial 1 finished with value: 0.9462497736151717 and parameters: {'iterations': 231, 'depth': 10, 'learning_rate': 0.031247251264037327, 'l2_leaf_reg': 0.053930277183791495, 'border_count': 219}. Best is trial 0 with value: 0.9479185532069028.


Training time: 54.15162444114685 seconds
Prediction time: 0.1235039234161377 seconds


[I 2025-03-05 11:12:57,665] Trial 2 finished with value: 0.946689606995938 and parameters: {'iterations': 834, 'depth': 7, 'learning_rate': 0.02416094989917, 'l2_leaf_reg': 0.011292150662893564, 'border_count': 162}. Best is trial 0 with value: 0.9479185532069028.


...

Training time: 183.7515320777893 seconds
Prediction time: 0.3712186813354492 seconds


[I 2025-03-05 14:51:59,472] Trial 98 finished with value: 0.9513596025976042 and parameters: {'iterations': 961, 'depth': 10, 'learning_rate': 0.10126452253974075, 'l2_leaf_reg': 0.03645882079478406, 'border_count': 182}. Best is trial 75 with value: 0.9535975783291506.


Training time: 205.25012636184692 seconds
Prediction time: 0.3560781478881836 seconds


[I 2025-03-05 14:55:13,363] Trial 99 finished with value: 0.952355695842281 and parameters: {'iterations': 914, 'depth': 10, 'learning_rate': 0.07145376811146116, 'l2_leaf_reg': 0.2602021323477133, 'border_count': 175}. Best is trial 75 with value: 0.9535975783291506.


Training time: 193.23074769973755 seconds
Prediction time: 0.6122791767120361 seconds
Best Hyperparameters for CatBoost: {'iterations': 983, 'depth': 10, 'learning_rate': 0.04472903424802527, 'l2_leaf_reg': 0.019935466076477358, 'border_count': 230}
Final Training time: 221.39059686660767 seconds
Final Test Accuracy for CatBoost: 0.9535975783291506
Final Prediction time: 0.6458010673522949 seconds
Modelo guardado en: /content/drive/MyDrive/Modelos/catboost_model_unsw.pkl
Modelo catboost_model cargado correctamente desde /content/drive/MyDrive/Modelos/catboost_model_unsw.pkl


#### SVM

In [ ]:
# Rutas para los archivos
CHECKPOINT_PATH = '/content/drive/MyDrive/optuna_svc_study.pkl'
TRAIN_TIME_PATH = '/content/drive/MyDrive/final_model_train_time_svc.txt'
BEST_PARAMS_PATH = '/content/drive/MyDrive/best_hyperparameters_svc.json'

# Definir la función objetivo para Optuna
def objective_svc(trial):
    # Parámetros a optimizar
    kernel = trial.suggest_categorical('kernel', ['rbf', 'sigmoid'])
    gamma = trial.suggest_float('gamma', 0.001, 10, log=True)

    # Entrenar el modelo
    svc = SVC(kernel=kernel, gamma=gamma)
    svc.fit(X_train, y_train)

    # Evaluar en conjunto de testing
    y_pred = svc.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy

# Verificar si hay un checkpoint previo
try:
    # Intentar cargar el checkpoint existente
    study = load(CHECKPOINT_PATH)
    print(f"Checkpoint cargado. Trials completados: {len(study.trials)}")
except FileNotFoundError:
    # Si no hay checkpoint, crear un nuevo estudio
    study = optuna.create_study(direction='maximize')
    print("No se encontró un checkpoint. Creando un nuevo estudio.")

# Calcular trials restantes
desired_trials = 10
completed_trials = len(study.trials)
remaining_trials = max(0, desired_trials - completed_trials)

if remaining_trials > 0:
    print(f"Completando los {remaining_trials} trials restantes...")
    study.optimize(objective_svc, n_trials=remaining_trials, callbacks=[lambda study, trial: dump(study, CHECKPOINT_PATH)])
else:
    print(f"Ya se completaron los {desired_trials} trials previstos.")

# Obtener los mejores hiperparámetros y la mejor puntuación
best_params = study.best_trial.params
best_score = study.best_value

# Guardar los mejores hiperparámetros en un archivo JSON
with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f)
print(f"Mejores hiperparámetros guardados en: {BEST_PARAMS_PATH}")

# Entrenar el modelo final con los mejores hiperparámetros
best_svc = SVC(kernel=best_params['kernel'], gamma=best_params['gamma'])

# Medir el tiempo de entrenamiento (si no está guardado)
try:
    with open(TRAIN_TIME_PATH, 'r') as f:
        svc_train = float(f.read())
        print(f"Training time (previamente guardado): {svc_train} segundos")
except FileNotFoundError:
    # Si no está guardado, calcularlo y guardarlo
    start_time = time.time()
    best_svc.fit(X_train, y_train)
    end_time = time.time()
    svc_train = end_time - start_time
    print("Training time (calculado): ", svc_train)
    with open(TRAIN_TIME_PATH, 'w') as f:
        f.write(str(svc_train))

# Medir el tiempo de predicción
start_time = time.time()
y_pred = best_svc.predict(X_test)
end_time = time.time()
svc_test = end_time - start_time
print("Testing time: ", svc_test)

# Evaluar y mostrar los resultados
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')  # Ajuste para varias clases si es necesario

print(f"F1 Score: {f1 * 100:.2f}%")
print(f"Accuracy Score: {accuracy * 100:.2f}%")

SVC_train, SVC_test = best_svc.score(X_train, y_train), best_svc.score(X_test, y_test)
print(f"Train Score: {SVC_train}")
print(f"Test Score: {SVC_test}")

# Imprimir los mejores hiperparámetros y la mejor puntuación
print("Best hyperparameters: ", best_params)
print("Best score: ", best_score)

# Guardar y mostrar las predicciones como DataFrames
y_val_pred1_split1 = pd.DataFrame(y_pred)
y_test_pred1_split1 = pd.DataFrame(best_svc.predict(X_test))

# Guardar el modelo entrenado
guardar_modelo(best_svc, 'svc_model')

# Cargar el modelo y evaluarlo
loaded_svc_model = cargar_modelo('svc_model')
result = loaded_svc_model.score(X_test, y_test)
print(f"Precisión del modelo cargado: {result}")


Checkpoint cargado. Trials completados: 10
Ya se completaron los 10 trials previstos.
Mejores hiperparámetros guardados en: /content/drive/MyDrive/best_hyperparameters_svc.json
Training time (calculado):  8187.012434720993
Testing time:  1255.2286128997803
F1 Score: 93.02%
Accuracy Score: 93.05%
Train Score: 0.9515332287341092
Test Score: 0.9304675170112028
Best hyperparameters:  {'kernel': 'rbf', 'gamma': 1.1740251511706965}
Best score:  0.9304675170112028
Modelo guardado en: /content/drive/MyDrive/Modelos/svc_model_unsw.pkl
Modelo svc_model cargado correctamente desde /content/drive/MyDrive/Modelos/svc_model_unsw.pkl
Precisión del modelo cargado: 0.9304675170112028


#### Redes Neuronales

In [ ]:

# Desactivar optimizaciones de OneDNN
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"


def create_model(trial):
    """
    Crea un modelo de red neuronal con hiperparámetros optimizados por Optuna.
    """
    # Definir los hiperparámetros de la red neuronal
    n_layers = trial.suggest_int('n_layers', 1, 5)
    model = Sequential()

    # Capa de entrada
    input_shape = X_train.shape[1]
    model.add(Input(shape=(input_shape,)))

    # Capas ocultas
    for i in range(n_layers):
        num_units = trial.suggest_int(f'n_units_l{i}', 8, 256)
        dropout_rate = trial.suggest_float(f'dropout_l{i}', 0.1, 0.6)
        model.add(Dense(num_units, activation='relu'))
        model.add(Dropout(dropout_rate))

    # Capa de salida para clasificación binaria
    model.add(Dense(1, activation='sigmoid'))

    # Definir el optimizador y la tasa de aprendizaje
    learning_rate = trial.suggest_float('learning_rate', 1e-6, 1e-1, log=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # Compilar el modelo
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def objective(trial):
    """
    Función objetivo de Optuna para encontrar los mejores hiperparámetros.
    """
    model = create_model(trial)

    # Entrenar el modelo
    start_time = time.time()
    batch_size = trial.suggest_int('batch_size', 16, 256)
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=batch_size, verbose=0)
    end_time = time.time()
    training_time = end_time - start_time
    print(f"Tiempo de entrenamiento: {training_time} segundos")

    # Evaluar el modelo en el conjunto de prueba
    score = model.evaluate(X_test, y_test, verbose=0)
    accuracy = score[1]  # 'accuracy' es el segundo valor de la salida
    return accuracy


# Crear el estudio de Optuna para maximizar la precisión
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Obtener los mejores hiperparámetros y la mejor puntuación
best_params = study.best_trial.params
best_score = study.best_trial.value
print(f"Mejores Hiperparámetros: {best_params}")
print(f"Mejor Score: {best_score}")

# Entrenar el modelo final con los mejores hiperparámetros
final_model = create_model(optuna.trial.FixedTrial(best_params))
final_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=best_params['batch_size'], verbose=0)

# Medir el tiempo de predicción
start_time = time.time()
y_test_pred_prob = final_model.predict(X_test)
y_test_pred = (y_test_pred_prob > 0.5).astype("int32")
end_time = time.time()
prediction_time = end_time - start_time
print(f"Tiempo de predicción: {prediction_time} segundos")

# Evaluar y mostrar los resultados
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Precisión en Test: {accuracy}")

# Mostrar las primeras predicciones en un DataFrame
y_test_pred_df = pd.DataFrame(y_test_pred, columns=['Prediction'])
print(y_test_pred_df.head())

# Guardar el modelo con TensorFlow (
model_path = "/content/drive/MyDrive/Modelos/simple_neural_network_model_unsw.keras"
final_model.save(model_path)  # Guardado en formato Keras
print(f"Modelo guardado correctamente en: {model_path}")

# Cargar el modelo guardado
loaded_model = tf.keras.models.load_model(model_path)
print("Modelo cargado correctamente")


[I 2025-03-05 18:26:11,299] A new study created in memory with name: no-name-fe7df884-6988-47a9-a9b6-a78dad9a51b6


Tiempo de entrenamiento: 201.50796127319336 segundos


[I 2025-03-05 18:29:35,531] Trial 0 finished with value: 0.9284494519233704 and parameters: {'n_layers': 1, 'n_units_l0': 23, 'dropout_l0': 0.5086573859375461, 'learning_rate': 0.0019954967204958334, 'batch_size': 19}. Best is trial 0 with value: 0.9284494519233704.


Tiempo de entrenamiento: 32.09718918800354 segundos


[I 2025-03-05 18:30:10,315] Trial 1 finished with value: 0.8946598768234253 and parameters: {'n_layers': 2, 'n_units_l0': 86, 'dropout_l0': 0.10249268174345458, 'n_units_l1': 141, 'dropout_l1': 0.5673662708813885, 'learning_rate': 0.04068867093078849, 'batch_size': 250}. Best is trial 0 with value: 0.9284494519233704.


...

Tiempo de entrenamiento: 185.1492509841919 segundos


[I 2025-03-05 21:25:42,136] Trial 98 finished with value: 0.9343096017837524 and parameters: {'n_layers': 3, 'n_units_l0': 205, 'dropout_l0': 0.21425834236534672, 'n_units_l1': 140, 'dropout_l1': 0.3494958738914349, 'n_units_l2': 37, 'dropout_l2': 0.30502915288362276, 'learning_rate': 0.000455400274504585, 'batch_size': 31}. Best is trial 91 with value: 0.9352668523788452.


Tiempo de entrenamiento: 193.06336903572083 segundos


[I 2025-03-05 21:29:00,533] Trial 99 finished with value: 0.9324597120285034 and parameters: {'n_layers': 3, 'n_units_l0': 206, 'dropout_l0': 0.17926431868056458, 'n_units_l1': 152, 'dropout_l1': 0.2811900198080341, 'n_units_l2': 38, 'dropout_l2': 0.3039918892585531, 'learning_rate': 0.0010919528474785045, 'batch_size': 30}. Best is trial 91 with value: 0.9352668523788452.


Mejores Hiperparámetros: {'n_layers': 2, 'n_units_l0': 224, 'dropout_l0': 0.21553282312284752, 'n_units_l1': 99, 'dropout_l1': 0.3476313890559869, 'learning_rate': 0.0003343114074901016, 'batch_size': 38}
Mejor Score: 0.9352668523788452
2416/2416 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step  
Tiempo de predicción: 5.250486612319946 segundos
Precisión en Test: 0.9337792036428553
   Prediction
0           0
1           1
2           1
3           0
4           0
Modelo guardado correctamente en: /content/drive/MyDrive/Modelos/simple_neural_network_model_unsw.keras
Modelo cargado correctamente


#### Random Forest

In [ ]:
# Obtener la ruta del directorio actual
base_dir = os.getcwd()

# Definir la ruta de la carpeta Modelos en Google Drive
models_dir = os.path.join(base_dir, 'drive', 'MyDrive', 'Modelos')

# Verificar la existencia del directorio de modelos
if not os.path.exists(models_dir):
    raise FileNotFoundError(f"El directorio {models_dir} no existe")

# Obtener todos los archivos en el directorio de modelos que terminan con '_unsw.pkl'
model_files = [f for f in os.listdir(models_dir) if f.endswith('_unsw.pkl')]

# Cargar los modelos dinámicamente usando el método cargar_modelo
models = {}
for model_file in model_files:
    model_name = model_file.replace('_unsw.pkl', '')  # Obtener el nombre base del modelo
    print(f"Cargando modelo: {model_name}")
    models[model_name] = cargar_modelo(model_name)  # Usar el método cargar_modelo
    print(f"Modelo {model_name} cargado correctamente.\n")

# Mostrar los modelos cargados
print("\nModelos cargados:")
for model_name in models:
    print(f"- {model_name}")


In [ ]:

# Configuraciones para Checkpoints y archivos
CHECKPOINT_PATH = '/content/drive/MyDrive/optuna_rf_study_unsw15.pkl'
TRAIN_TIME_PATH = '/content/drive/MyDrive/rf_train_time_unsw15.txt'
BEST_PARAMS_PATH = '/content/drive/MyDrive/rf_best_hyperparameters_unsw15.json'

# Convertir X_test a un DataFrame si es necesario
if isinstance(X_test, np.ndarray):
    X_test = pd.DataFrame(X_test, columns=[f'feature_{i}' for i in range(X_test.shape[1])])

# Generar predicciones con los modelos ya entrenados (manteniendo los nombres originales)
decision_tree_pred = models['decision_tree_model'].predict(X_test)
logistic_regression_pred = models['logistic_regression_model'].predict(X_test)
svc_pred = models['svc_model'].predict(X_test)
knn_pred = models['knn_model'].predict(X_test)

# Crear un nuevo conjunto de datos combinando las características originales y las predicciones de los modelos
stacked_features = X_test.copy()
stacked_features['decision_tree_pred'] = decision_tree_pred
stacked_features['logistic_regression_pred'] = logistic_regression_pred
stacked_features['svc_pred'] = svc_pred
stacked_features['knn_pred'] = knn_pred

# Definir la función objetivo para la optimización de hiperparámetros
def objective_randomforest(trial):
    n_estimators = trial.suggest_int("n_estimators", 200, 1500)
    max_depth = trial.suggest_int("max_depth", 10, 100)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 4)

    # Crear y entrenar el modelo de Random Forest utilizando las características apiladas
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    rf.fit(stacked_features, y_test)

    # Realizar predicciones y calcular la precisión
    y_pred = rf.predict(stacked_features)
    acc = accuracy_score(y_test, y_pred)
    return acc

# Cargar el checkpoint si existe, o crear un nuevo estudio
try:
    study = joblib.load(CHECKPOINT_PATH)
    print(f"Checkpoint cargado. Trials completados: {len(study.trials)}")
except FileNotFoundError:
    study = optuna.create_study(direction='maximize')
    print("No se encontró un checkpoint. Creando un nuevo estudio.")

# Completar trials restantes
desired_trials = 100
completed_trials = len(study.trials)
remaining_trials = max(0, desired_trials - completed_trials)

if remaining_trials > 0:
    print(f"Completando los {remaining_trials} trials restantes...")
    study.optimize(objective_randomforest, n_trials=remaining_trials,
                   callbacks=[lambda study, trial: joblib.dump(study, CHECKPOINT_PATH)])
else:
    print(f"Ya se completaron los {desired_trials} trials previstos.")

# Guardar los mejores hiperparámetros
best_params = study.best_trial.params
with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f)
print(f"Mejores hiperparámetros guardados en: {BEST_PARAMS_PATH}")

# Entrenar el modelo final con los mejores hiperparámetros
rf_best = RandomForestClassifier(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    random_state=42
)

# Medir el tiempo de entrenamiento
try:
    with open(TRAIN_TIME_PATH, 'r') as f:
        rf_best_train_time = float(f.read())
        print(f"Tiempo de entrenamiento (previamente guardado): {rf_best_train_time} segundos")
except FileNotFoundError:
    start_time = time.time()
    rf_best.fit(stacked_features, y_test)
    end_time = time.time()
    rf_best_train_time = end_time - start_time
    print("Tiempo de entrenamiento (calculado):", rf_best_train_time)
    with open(TRAIN_TIME_PATH, 'w') as f:
        f.write(str(rf_best_train_time))

# Medir el tiempo de predicción
start_time = time.time()
final_predictions = rf_best.predict(stacked_features)
end_time = time.time()
rf_best_test_time = end_time - start_time
print("Tiempo de predicción:", rf_best_test_time)

# Evaluar el modelo
accuracy = accuracy_score(y_test, final_predictions)
f1 = f1_score(y_test, final_predictions, average='weighted')

print(f"Stacked Model Accuracy: {accuracy * 100:.2f}%")
print(f"Stacked Model F1 Score: {f1 * 100:.2f}%")

# Guardar el modelo final entrenado
guardar_modelo(rf_best, 'rf_stacking_model_optimized')

# Cargar y verificar el modelo guardado
loaded_rf_stacking = cargar_modelo('rf_stacking_model_optimized')
print("Stacking and Optimized Model loaded successfully")


[I 2025-03-09 23:01:06,503] A new study created in memory with name: no-name-15b358a0-dcee-4ffc-89c9-69284a4971c3


No se encontró un checkpoint. Creando un nuevo estudio.
Completando los 100 trials restantes...


[I 2025-03-09 23:04:59,874] Trial 0 finished with value: 0.9642182608470673 and parameters: {'n_estimators': 1475, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.9642182608470673.
[I 2025-03-09 23:07:50,958] Trial 1 finished with value: 0.9682673152052987 and parameters: {'n_estimators': 1074, 'max_depth': 72, 'min_samples_split': 7, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.9682673152052987.
[I 2025-03-09 23:09:14,807] Trial 2 finished with value: 0.9763783601976663 and parameters: {'n_estimators': 502, 'max_depth': 26, 'min_samples_split': 8, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.9763783601976663.
[I 2025-03-09 23:09:59,050] Trial 3 finished with value: 0.9682931877571085 and parameters: {'n_estimators': 283, 'max_depth': 46, 'min_samples_split': 6, 'min_samples_leaf': 4}. Best is trial 2 with value: 0.9763783601976663.
[I 2025-03-09 23:11:16,259] Trial 4 finished with value: 0.9703888644537011 and parameters

Mejores hiperparámetros guardados en: /content/drive/MyDrive/rf_best_hyperparameters_unsw15.json
Tiempo de entrenamiento (calculado): 97.25264859199524
Tiempo de predicción: 8.849186182022095
Stacked Model Accuracy: 99.85%
Stacked Model F1 Score: 99.85%
Modelo guardado en: /content/drive/MyDrive/Modelos/rf_stacking_model_optimized_unsw.pkl
Modelo rf_stacking_model_optimized cargado correctamente desde /content/drive/MyDrive/Modelos/rf_stacking_model_optimized_unsw.pkl
Stacking and Optimized Model loaded successfully
